# DWD RADKLIM / RADOLAN — catalog explorer (no network)

Explore the `earthlens.radklim` backend offline: the four products, how a request enumerates the raw granules it would download, and a real RADOLAN granule read + plotted with `pyramids`. Everything here runs deterministically at docs-build time — no DWD access.

RADKLIM is DWD's gauge-adjusted radar precipitation over Germany: the reprocessed climatology **RADKLIM** (statistics) and the operational near-real-time **RADOLAN** stream. See the [API reference](../../reference/radklim/introduction.md).

## Setup

Imports up front. `earthlens.radklim.Catalog` loads the bundled product catalog (no network); the `EarthLens` facade builds a request; `pyramids` reads the granule; `pandas` / `matplotlib` render the tables and figure. `DATA` is a notebook-relative path to the docs example-data folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from pyramids.dataset import Dataset, GeoReference

from earthlens.core import EarthLens
from earthlens.radklim import Catalog
from earthlens.radklim._helpers import operational_granule_url

DATA = Path('../../../examples/data/radklim')

## The product catalog

Instantiate the catalog and read its licence and grid. RADKLIM is DWD open geodata under **CC-BY-4.0 / GeoNutzV** — attribution to *Deutscher Wetterdienst (DWD)* is required.

In [ ]:
cat = Catalog()
print('products :', cat.products())
print('license  :', cat.license)
print('grid     :', cat.grid['id'])

## Products at a glance

The four products split across two streams. **reproc** (RADKLIM) is the climatology — one yearly NetCDF archive per year, full 2001- record, use it for statistics. **operational** (RADOLAN) is near-real-time — per-timestamp granules on a rolling ~2-day window. Each comes in an hourly (`RW`) and a 5-min (`YW`) flavour.

In [ ]:
rows = [
    {
        'product': k,
        'stream': p.stream,
        'cadence': p.cadence,
        'format': p.default_format,
        'retention_days': p.retention_days or '-',
    }
    for k, p in cat.datasets.items()
]
pd.DataFrame(rows).set_index('product')

**RADKLIM-YW (5-min, 1 km) is the single most useful dataset for German sub-hourly extreme rainfall.** The record is ~25 years — excellent for event structure, short for long return periods. For return-period work prefer RADKLIM over the inhomogeneous operational RADOLAN stream.

## The download plan (reproc)

Build a request through the `EarthLens` facade with `dataset='radklim-yw'`. The reprocessing has no finer addressable unit than the **year**, so a `[start, end]` window maps to one yearly `.tar.gz` NetCDF archive per year. `_search()` returns that plan without touching the network — a cheap dry-run of what `download()` would fetch.

In [ ]:
lens = EarthLens(
    data_source='radklim',
    dataset='radklim-yw',
    start='2020-06-01',
    end='2022-06-01',
    lat_lim=[47.0, 55.0],
    lon_lim=[6.0, 15.0],
    path='.',
)
plan = lens.datasource._search()
pd.DataFrame([{'id': p.id, 'file': p.href.rsplit('/', 1)[-1]} for p in plan])

Three years in the window → three yearly archives. These are large (YW ~13.5 GB/yr, RW ~836 MB/yr), so `download()` streams them to disk — this notebook only inspects the plan.

## Operational granule URLs

The operational stream is addressed per timestamp instead. A granule name carries a `YYMMDDHHMM` stamp; the helper builds its URL. earthlens reads the stream's directory listing and keeps the granules inside the request window (behind the ~2-day retention guard).

In [ ]:
operational_granule_url('yw', 'raa01-yw_10000-2608101820-dwd---bin.hdf5')

## Read a real RADOLAN granule with pyramids

earthlens returns raw granule paths; reading them is `pyramids`' job. A small **real** operational RADOLAN-RW HDF5 granule ships with the docs. It opens directly on the fixed RADOLAN polar-stereographic grid over Germany.

In [ ]:
granule = DATA / 'raa01-rw_10000-2608101830-dwd---bin.hdf5'
ds = Dataset.read_file(granule)
print('shape:', ds.shape)
print('crs  :', ds.crs.split('PROJECTION')[-1][:40], '...')

## Plot the precipitation field

Read the array and mask the negative no-data cells, then show the hourly precipitation over the RADOLAN grid. The Germany outline of the composite is clearly visible.

In [ ]:
arr = ds.read_array()
band = arr[0] if getattr(arr, 'ndim', 2) == 3 else arr
precip = np.where(band < 0, np.nan, band)

rain = Dataset.from_array(
    precip,
    no_data_value=np.nan,
    geo_ref=GeoReference(geo=ds.geotransform, epsg=ds.epsg),
)
glyph = rain.plot(
    cmap='Blues',
    vmax=np.nanpercentile(precip, 99),
    title='RADOLAN-RW operational granule — (c) Deutscher Wetterdienst (DWD)',
)
glyph.cbar.set_label('hourly precipitation (raw units)')

## Takeaway

- Four products, two streams: **RADKLIM** (reproc, yearly NetCDF archives, statistics) and **RADOLAN** (operational, per-timestamp HDF5, near-real-time).
- A request enumerates raw granules — yearly archives for reproc, in-window timestamps for operational — and `download()` returns their paths.
- Reading (NetCDF / HDF5) is `pyramids`; earthlens never imports `wradlib` / `xarray` / `netCDF4`.
- The data is DWD open geodata (**CC-BY-4.0 / GeoNutzV**) — always credit *Deutscher Wetterdienst (DWD)*.